# Packages & Libraries

In [1]:
from statsforecast import StatsForecast
from mlforecast import MLForecast
from mlforecast.target_transforms import Differences
from mlforecast.utils import PredictionIntervals
from window_ops.expanding import expanding_mean
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.linear_model import Lasso, LinearRegression, Ridge
from sklearn.neural_network import MLPRegressor
from sklearn.ensemble import RandomForestRegressor
from utilsforecast.plotting import plot_series

import mlflow
import mlforecast.flavor

import pandas as pd
import numpy as np
import requests
import json
import os
import datetime
from statistics import mean

/Users/aizhanm/Desktop/forecasting_service/.venv_frcst/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Data

In [2]:
DATA_TRAIN_FILE_PATH = "../data/train.csv"
PRODUCT_NAME = "BEVERAGES"
STORE_NUMBER = 1

In [3]:
ds = pd.read_csv(DATA_TRAIN_FILE_PATH)
ds = ds[(ds["family"] == PRODUCT_NAME) & (ds["store_nbr"] == STORE_NUMBER)]
ds["unique_id"] = 1  # ds["store_nbr"].astype(str) + "_" + ds["family"]
ds = ds[["date", "sales", "unique_id"]]
ds["date"] = pd.to_datetime(ds["date"], format="%Y-%m-%d")
ds = ds.sort_values("date")
ds = ds.rename(columns={"date": "ds", "sales": "y"})

In [4]:
fig = StatsForecast.plot(ds, engine="plotly")
fig.show()

In [5]:
ds["ds"] = pd.to_datetime(ds["ds"])

g = ds.sort_values("ds")

missing = pd.date_range(
    g["ds"].min(),
    g["ds"].max(),
    freq="D"
).difference(g["ds"])

missing = pd.DataFrame({
    "ds": pd.to_datetime(missing),
    "y": [0]*len(missing),
    "unique_id": [1]*len(missing)
})

ds = pd.concat([ds, missing], ignore_index=True).sort_values("ds")

In [6]:
train_size = int(len(ds) * 0.8)
train, test = ds.iloc[:train_size], ds.iloc[train_size:]

# Model

In [7]:
ml_models = {
    "lightGBM": LGBMRegressor(n_estimators=500, verbosity=-1),
    "xgboost": XGBRegressor(),
    "linear_regression": LinearRegression(),
    "lasso": Lasso(),
    "ridge": Ridge()
}

In [8]:
params = {
    "freq": "D",
    "lags": list(range(1, 56)),
    "date_features": ["year", "month", "day"]
}

In [9]:
mlf = MLForecast(
    models=ml_models,
    freq=params["freq"],
    lags=params["lags"],
    date_features=params["date_features"]
)

mlf.fit(train)

MLForecast(models=[lightGBM, xgboost, linear_regression, lasso, ridge], freq=D, lag_features=['lag1', 'lag2', 'lag3', 'lag4', 'lag5', 'lag6', 'lag7', 'lag8', 'lag9', 'lag10', 'lag11', 'lag12', 'lag13', 'lag14', 'lag15', 'lag16', 'lag17', 'lag18', 'lag19', 'lag20', 'lag21', 'lag22', 'lag23', 'lag24', 'lag25', 'lag26', 'lag27', 'lag28', 'lag29', 'lag30', 'lag31', 'lag32', 'lag33', 'lag34', 'lag35', 'lag36', 'lag37', 'lag38', 'lag39', 'lag40', 'lag41', 'lag42', 'lag43', 'lag44', 'lag45', 'lag46', 'lag47', 'lag48', 'lag49', 'lag50', 'lag51', 'lag52', 'lag53', 'lag54', 'lag55'], date_features=['year', 'month', 'day'], num_threads=1)

# Backtesting

In [10]:
#
partitions = 10 # size of partitions
step_size = 24 # window shift size
h = 72 # size of testing partitions

n_windows = 5
method = "conformal_distribution"
pi = PredictionIntervals(h=h, n_windows=n_windows, method=method)
levels = [95]

In [11]:
bkt_df = mlf.cross_validation(
    df = train,
    h = h,
    step_size=step_size,
    n_windows=partitions,
    prediction_intervals=pi,
    level=levels
)

In [12]:
bkt_df.head()

,unique_id,ds,cutoff,y,lightGBM,xgboost,linear_regression,lasso,ridge,lightGBM-lo-95,lightGBM-hi-95,xgboost-lo-95,xgboost-hi-95,linear_regression-lo-95,linear_regression-hi-95,lasso-lo-95,lasso-hi-95,ridge-lo-95,ridge-hi-95
0,1,2015-11-29,2015-11-28,1070.0,1042.754166,1013.568237,1197.485471,1195.065379,1197.392599,256.816119,1828.692212,38.063370,1989.073105,506.484125,1888.486817,493.410055,1896.720704,504.565430,1890.219769
1,1,2015-11-30,2015-11-28,2342.0,2041.674510,1981.151245,1994.031081,1990.449698,1993.892027,1120.465351,2962.883669,1315.927103,2646.375388,1176.963825,2811.098336,1163.937867,2816.961528,1175.223615,2812.560439
2,1,2015-12-01,2015-11-28,2462.0,2056.469009,1806.922363,2054.104514,2048.726287,2053.915391,1264.259438,2848.678580,1137.117725,2476.727002,1202.881435,2905.327593,1231.112298,2866.340276,1208.765781,2899.065000
3,1,2015-12-02,2015-11-28,2520.0,2389.871105,2279.667236,2488.622850,2482.115711,2488.398833,1453.012104,3326.730106,1672.117664,2887.216809,1457.973918,3519.271782,1474.755765,3489.475657,1461.993307,3514.804360
4,1,2015-12-03,2015-11-28,2186.0,1845.132887,1925.597046,2231.952538,2224.262236,2231.688489,1120.412122,2569.853651,1334.921069,2516.273022,1117.979623,3345.925454,1160.251798,3288.272674,1126.589383,3336.787596


In [13]:
cutoff = bkt_df["cutoff"].unique()
partitions_mapping = pd.DataFrame({
    "cutoff": cutoff,
    "partition": range(1, len(cutoff)+1)
    })
print(partitions_mapping)

      cutoff  partition
0 2015-11-28          1
1 2015-12-22          2
2 2016-01-15          3
3 2016-02-08          4
4 2016-03-03          5
5 2016-03-27          6
6 2016-04-20          7
7 2016-05-14          8
8 2016-06-07          9
9 2016-07-01         10


In [14]:
model_label = ["lightGBM", "xgboost", "linear_regression", "lasso", "ridge"]
model_name = ['LGBMRegressor', 'XGBRegressor', 'LinearRegression', 'Lasso', 'Ridge']

models_mapping = pd.DataFrame({"model_label": model_label, "model_name": model_name})
models_mapping

,model_label,model_name
0,lightGBM,LGBMRegressor
1,xgboost,XGBRegressor
2,linear_regression,LinearRegression
3,lasso,Lasso
4,ridge,Ridge


In [15]:
models = list(ml_models.keys())

bkt_long = pd.melt(
    bkt_df,
    id_vars=["unique_id", "ds", "cutoff", "y"],
    value_vars=models + [f"{model}-lo-95" for model in models] + [f"{model}-hi-95" for model in models],
    var_name="model_label",
    value_name="value",
)
bkt_long.head()

,unique_id,ds,cutoff,y,model_label,value
0,1,2015-11-29,2015-11-28,1070.0,lightGBM,1042.754166
1,1,2015-11-30,2015-11-28,2342.0,lightGBM,2041.674510
2,1,2015-12-01,2015-11-28,2462.0,lightGBM,2056.469009
3,1,2015-12-02,2015-11-28,2520.0,lightGBM,2389.871105
4,1,2015-12-03,2015-11-28,2186.0,lightGBM,1845.132887


In [16]:
def split_model_confidence(model_name):
    if "-lo-95" in model_name:
        return model_name.replace("-lo-95", ""), "lower"
    elif "-hi-95" in model_name:
        return model_name.replace("-hi-95", ""), "upper"
    else:
        return model_name, "forecast"

bkt_long["model_label"], bkt_long["type"] = zip(*bkt_long["model_label"].map(split_model_confidence))

In [17]:
bkt_long = bkt_long.merge(partitions_mapping, how="left", on=["cutoff"])

bkt = (bkt_long
.pivot(index=["unique_id", "ds", "model_label", "partition", "y"],
       columns="type", values="value")
.reset_index()
.merge(models_mapping, how="left", on=["model_label"])
       )

In [18]:
bkt

,unique_id,ds,model_label,partition,y,forecast,lower,upper,model_name
0,1,2015-11-29,lasso,1,1070.0,1195.065379,493.410055,1896.720704,Lasso
1,1,2015-11-29,lightGBM,1,1070.0,1042.754166,256.816119,1828.692212,LGBMRegressor
2,1,2015-11-29,linear_regression,1,1070.0,1197.485471,506.484125,1888.486817,LinearRegression
3,1,2015-11-29,ridge,1,1070.0,1197.392599,504.565430,1890.219769,Ridge
4,1,2015-11-29,xgboost,1,1070.0,1013.568237,38.063370,1989.073105,XGBRegressor
...,...,...,...,...,...,...,...,...,...
3595,1,2016-09-11,lasso,10,864.0,1545.187572,-365.830228,3456.205372,Lasso
3596,1,2016-09-11,lightGBM,10,864.0,775.988301,-1893.854605,3445.831207,LGBMRegressor
3597,1,2016-09-11,linear_regression,10,864.0,1568.759059,-314.940538,3452.458655,LinearRegression
3598,1,2016-09-11,ridge,10,864.0,1567.915457,-316.765900,3452.596813,Ridge


In [19]:
def mape(y, yhat):
    mape = mean(abs(y - yhat)/ y) 
    return mape

def rmse(y, yhat):
    rmse = (mean((y - yhat) ** 2 )) ** 0.5
    return rmse

def coverage(y, lower, upper):
    coverage = sum((y <= upper) & (y >= lower)) / len(y)
    return coverage

def score(df):
    mape_score = mape(y = df["y"], yhat = df["forecast"])
    rmse_score = rmse(y = df["y"], yhat = df["forecast"])
    coverage_score = coverage(y = df["y"], lower = df["lower"], upper = df["upper"])
    cols = ["mape", "rmse", "coverage"]
    df = pd.Series([mape_score, rmse_score,  coverage_score], index=cols)
    return df

In [20]:
# Create score_df
score_df = (bkt
            .groupby(["unique_id", "model_label", "model_name", "partition"])
            [["unique_id", "model_label", "model_name", "partition", "y", "forecast", "lower", "upper"]]
            .apply(score)
            .reset_index()
)

score_df

,unique_id,model_label,model_name,partition,mape,rmse,coverage
0,1,lasso,Lasso,1,inf,503.472799,0.972222
1,1,lasso,Lasso,2,inf,551.932657,0.958333
2,1,lasso,Lasso,3,0.253773,493.956554,0.944444
3,1,lasso,Lasso,4,0.198625,704.295927,0.750000
4,1,lasso,Lasso,5,0.142461,576.104212,0.875000
5,1,lasso,Lasso,6,0.157070,592.725478,0.902778
6,1,lasso,Lasso,7,0.385111,761.460520,0.777778
7,1,lasso,Lasso,8,0.162536,309.290967,0.972222
8,1,lasso,Lasso,9,0.152944,294.656045,0.930556
9,1,lasso,Lasso,10,0.190546,340.007222,0.916667


# MLFlow

In [21]:
tags = {
    "h":h,
    "step_size": step_size,
    "partitions": partitions,
    "intervals_type": "ConformalIntervals",
    "intervals_h": h,
    "intervals_n_windows": n_windows,
    "intervals_method": "conformal_distribution",
    "levels": levels
}

In [24]:
experiment_name = "ml_forecast_multimodel_new"
# mlflow_path = "file:///mlruns"
try:
    mlflow.create_experiment(name=experiment_name)
    meta = mlflow.get_experiment_by_name(experiment_name)
    print(f"Experiment '{experiment_name}' created with ID: {meta.experiment_id}")
except:
    meta = mlflow.get_experiment_by_name(experiment_name)
    print(f"Experiment '{experiment_name}' already exists with ID: {meta.experiment_id}")

run_time = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

Experiment 'ml_forecast_multimodel_new' created with ID: 6


In [29]:
for index, row in score_df.iterrows():
    run_name = row["model_label"] + "-" + run_time
    with mlflow.start_run(experiment_id=meta.experiment_id, run_name=run_name,
                          tags={"type": "backtesting", "partition": row["partition"],
                                "unique_id": row["unique_id"], "model_label": row["model_label"],
                                "model_name": row["model_name"], "run_name": run_name}) as run:
        model_params = ml_models[row["model_label"]].get_params()
        model_params["model_name"] = row["model_name"]
        model_params["model_label"] = row["model_label"]
        model_params["partition"] = row["partition"]
        model_params["lags"] = list(range(1, 56))
        model_params["date_features"] = ["year", "month", "day"]
        mlflow.log_params(model_params)
        mlflow.log_metric("mape", row["mape"])
        mlflow.log_metric("rmse", row["rmse"])
        mlflow.log_metric("coverage", row["coverage"])